In [ ]:
!pip -q install numpy pandas
print("install done")


install done


In [ ]:
%%writefile /content/qualcomm_3dmm_to_arkit52.py
from __future__ import annotations

import numpy as np

ARKIT_52_BLENDSHAPES = (
    "browDownLeft", "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight",
    "cheekPuff", "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft", "eyeBlinkRight",
    "eyeLookDownLeft", "eyeLookDownRight", "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft",
    "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight", "eyeSquintLeft", "eyeSquintRight",
    "eyeWideLeft", "eyeWideRight", "jawForward", "jawLeft", "jawOpen", "jawRight", "mouthClose",
    "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft", "mouthFrownRight", "mouthFunnel",
    "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft", "mouthPressRight",
    "mouthPucker", "mouthRight", "mouthRollLower", "mouthRollUpper", "mouthShrugLower",
    "mouthShrugUpper", "mouthSmileLeft", "mouthSmileRight", "mouthStretchLeft", "mouthStretchRight",
    "mouthUpperUpLeft", "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight", "tongueOut",
)

LEFT_BROW = (17, 18, 19, 20, 21)
RIGHT_BROW = (22, 23, 24, 25, 26)
LEFT_EYE = (36, 37, 38, 39, 40, 41)
RIGHT_EYE = (42, 43, 44, 45, 46, 47)


def clamp01(x):
    return float(np.clip(x, 0.0, 1.0))


def dist(points, a, b):
    return float(np.linalg.norm(points[a] - points[b]))


def eye_open(points, idx):
    a, b, c, d, e, f = idx
    width = max(dist(points, a, d), 1e-6)
    height = 0.5 * (dist(points, b, f) + dist(points, c, e))
    return height / width


def split_output(output):
    vec = np.asarray(output, dtype=np.float32).reshape(-1)
    if vec.size != 264:
        raise ValueError(f"Qualcomm output must have length 264, got {vec.size}")
    return {
        "identity": vec[:219].copy(),
        "expression": vec[219:258].copy(),
        "pitch": float(vec[258]),
        "yaw": float(vec[259]),
        "roll": float(vec[260]),
        "tx": float(vec[261]),
        "ty": float(vec[262]),
        "f": float(vec[263]),
    }


def reconstruct_68(output, mean_face, shape_basis, blendshape_basis):
    c = split_output(output)
    face = np.asarray(mean_face, dtype=np.float32).reshape(204, 1)
    basis_id = np.asarray(shape_basis, dtype=np.float32).reshape(204, 219)
    basis_exp = np.asarray(blendshape_basis, dtype=np.float32).reshape(204, 39)

    alpha_id = (c["identity"] * 3.0).reshape(219, 1)
    alpha_exp = (c["expression"] * 0.5 + 0.5).reshape(39, 1)
    pitch = c["pitch"] * np.pi / 2.0
    yaw = c["yaw"] * np.pi / 2.0
    roll = c["roll"] * np.pi / 2.0
    tx = c["tx"] * 60.0
    ty = c["ty"] * 60.0
    tz = 500.0
    focal = c["f"] * 150.0 + 450.0

    p = np.array([[1, 0, 0], [0, np.cos(-np.pi), -np.sin(-np.pi)], [0, np.sin(-np.pi), np.cos(-np.pi)]], dtype=np.float32)
    r_roll = np.array([[np.cos(-roll), -np.sin(-roll), 0], [np.sin(-roll), np.cos(-roll), 0], [0, 0, 1]], dtype=np.float32)
    r_yaw = np.array([[np.cos(-yaw), 0, np.sin(-yaw)], [0, 1, 0], [-np.sin(-yaw), 0, np.cos(-yaw)]], dtype=np.float32)
    r_pitch = np.array([[1, 0, 0], [0, np.cos(-pitch), -np.sin(-pitch)], [0, np.sin(-pitch), np.cos(-pitch)]], dtype=np.float32)
    rot = r_yaw @ r_pitch @ p @ r_roll

    verts = (face + basis_id @ alpha_id + basis_exp @ alpha_exp).reshape(68, 3) @ rot.T
    verts[:, 0] += tx
    verts[:, 1] += ty
    verts[:, 2] += tz
    pts2d = verts[:, :2] * focal / tz
    return pts2d.astype(np.float32), {"pitch": pitch, "yaw": yaw, "roll": roll}


def extract_metrics(points, mirror_input=False):
    if mirror_input:
        l_eye, r_eye = LEFT_EYE, RIGHT_EYE
        l_brow, r_brow = LEFT_BROW, RIGHT_BROW
        l_corner, r_corner = 48, 54
        l_nose, r_nose = 31, 35
        l_upper, r_upper = 49, 53
        l_lower, r_lower = 59, 55
        l_inner_mouth, r_inner_mouth = 67, 65
    else:
        l_eye, r_eye = RIGHT_EYE, LEFT_EYE
        l_brow, r_brow = RIGHT_BROW, LEFT_BROW
        l_corner, r_corner = 54, 48
        l_nose, r_nose = 35, 31
        l_upper, r_upper = 53, 49
        l_lower, r_lower = 55, 59
        l_inner_mouth, r_inner_mouth = 65, 67

    face_width = max(dist(points, 0, 16), 1e-6)
    face_height = max(abs(float(points[8, 1] - points[27, 1])), face_width * 0.6, 1e-6)
    mouth_center = 0.5 * (points[51] + points[57])
    left_eye_center = np.mean(points[np.asarray(l_eye)], axis=0)
    right_eye_center = np.mean(points[np.asarray(r_eye)], axis=0)

    out = {
        "face_width": face_width,
        "face_height": face_height,
        "left_eye_open": eye_open(points, l_eye),
        "right_eye_open": eye_open(points, r_eye),
        "mouth_open_inner": (dist(points, 61, 67) + dist(points, 62, 66) + dist(points, 63, 65)) / (3.0 * face_height),
        "mouth_open_outer": (dist(points, 50, 58) + dist(points, 51, 57) + dist(points, 52, 56)) / (3.0 * face_height),
        "mouth_width": dist(points, 48, 54) / face_width,
        "mouth_center_x": float(mouth_center[0]),
        "nose_x": float(points[33, 0]),
        "chin_x": float(points[8, 0]),
        "chin_drop": float(points[8, 1] - points[33, 1]) / face_height,
        "upper_lip_thickness": dist(points, 51, 62) / face_height,
        "lower_lip_thickness": dist(points, 57, 66) / face_height,
        "left_corner_raise": float(mouth_center[1] - points[l_corner, 1]) / face_height,
        "right_corner_raise": float(mouth_center[1] - points[r_corner, 1]) / face_height,
        "left_corner_stretch": float(abs(points[l_corner, 0] - mouth_center[0])) / face_width,
        "right_corner_stretch": float(abs(points[r_corner, 0] - mouth_center[0])) / face_width,
        "left_outer_brow_gap": float(left_eye_center[1] - points[l_brow[0], 1]) / face_height,
        "left_inner_brow_gap": float(left_eye_center[1] - points[l_brow[-1], 1]) / face_height,
        "right_inner_brow_gap": float(right_eye_center[1] - points[r_brow[0], 1]) / face_height,
        "right_outer_brow_gap": float(right_eye_center[1] - points[r_brow[-1], 1]) / face_height,
        "left_upper_nose_gap": float(points[l_upper, 1] - points[l_nose, 1]) / face_height,
        "right_upper_nose_gap": float(points[r_upper, 1] - points[r_nose, 1]) / face_height,
        "left_lower_chin_gap": float(points[8, 1] - points[l_lower, 1]) / face_height,
        "right_lower_chin_gap": float(points[8, 1] - points[r_lower, 1]) / face_height,
        "left_mouth_press_gap": dist(points, l_upper, l_inner_mouth) / face_height,
        "right_mouth_press_gap": dist(points, r_upper, r_inner_mouth) / face_height,
    }
    out["inner_brow_gap"] = 0.5 * (out["left_inner_brow_gap"] + out["right_inner_brow_gap"])
    return out


def delta(metrics, neutral, key, scale):
    return (float(metrics[key]) - float(neutral[key])) / max(scale, 1e-6)


def metrics_to_shapes(metrics, neutral, pose, use_head_pose_as_eye_gaze=False):
    mouth_open_abs = clamp01((float(metrics["mouth_open_inner"]) - 0.010) / 0.070)
    mouth_width_abs = clamp01((float(metrics["mouth_width"]) - 0.34) / 0.22)
    left_blink_abs = clamp01((0.30 - float(metrics["left_eye_open"])) / 0.18)
    right_blink_abs = clamp01((0.30 - float(metrics["right_eye_open"])) / 0.18)
    left_wide_abs = clamp01((float(metrics["left_eye_open"]) - 0.32) / 0.10)
    right_wide_abs = clamp01((float(metrics["right_eye_open"]) - 0.32) / 0.10)

    left_blink = max(left_blink_abs, clamp01(-delta(metrics, neutral, "left_eye_open", 0.08)))
    right_blink = max(right_blink_abs, clamp01(-delta(metrics, neutral, "right_eye_open", 0.08)))
    jaw_open = max(mouth_open_abs, clamp01(delta(metrics, neutral, "mouth_open_inner", 0.060)), clamp01(delta(metrics, neutral, "chin_drop", 0.18)))
    mouth_smile_left = max(clamp01((float(metrics["left_corner_raise"]) - 0.010) / 0.080), clamp01(delta(metrics, neutral, "left_corner_raise", 0.050)))
    mouth_smile_right = max(clamp01((float(metrics["right_corner_raise"]) - 0.010) / 0.080), clamp01(delta(metrics, neutral, "right_corner_raise", 0.050)))
    mouth_frown_left = max(clamp01((-float(metrics["left_corner_raise"]) - 0.005) / 0.080), clamp01(-delta(metrics, neutral, "left_corner_raise", 0.050)))
    mouth_frown_right = max(clamp01((-float(metrics["right_corner_raise"]) - 0.005) / 0.080), clamp01(-delta(metrics, neutral, "right_corner_raise", 0.050)))
    left_outer_up = max(clamp01((float(metrics["left_outer_brow_gap"]) - 0.09) / 0.08), clamp01(delta(metrics, neutral, "left_outer_brow_gap", 0.05)))
    right_outer_up = max(clamp01((float(metrics["right_outer_brow_gap"]) - 0.09) / 0.08), clamp01(delta(metrics, neutral, "right_outer_brow_gap", 0.05)))
    brow_inner_up = max(clamp01((float(metrics["inner_brow_gap"]) - 0.095) / 0.08), clamp01(delta(metrics, neutral, "inner_brow_gap", 0.05)))
    brow_down_left = max(clamp01((0.085 - float(metrics["left_inner_brow_gap"])) / 0.060), clamp01(-delta(metrics, neutral, "left_inner_brow_gap", 0.04))) * (1.0 - 0.35 * left_blink)
    brow_down_right = max(clamp01((0.085 - float(metrics["right_inner_brow_gap"])) / 0.060), clamp01(-delta(metrics, neutral, "right_inner_brow_gap", 0.04))) * (1.0 - 0.35 * right_blink)
    mouth_pucker = max(clamp01((0.43 - float(metrics["mouth_width"])) / 0.18) * clamp01((0.050 - jaw_open) / 0.050), clamp01(-delta(metrics, neutral, "mouth_width", 0.12)))
    mouth_funnel = clamp01((0.48 - float(metrics["mouth_width"])) / 0.20) * clamp01((float(metrics["mouth_open_outer"]) - 0.02) / 0.07)
    mouth_stretch_left = max(clamp01((float(metrics["left_corner_stretch"]) - 0.17) / 0.10), clamp01(delta(metrics, neutral, "left_corner_stretch", 0.06)))
    mouth_stretch_right = max(clamp01((float(metrics["right_corner_stretch"]) - 0.17) / 0.10), clamp01(delta(metrics, neutral, "right_corner_stretch", 0.06)))
    mouth_center_shift = float(metrics["mouth_center_x"] - metrics["nose_x"]) / max(float(metrics["face_width"]), 1e-6)
    jaw_shift = mouth_center_shift + 0.55 * (float(metrics["chin_x"] - metrics["nose_x"]) / max(float(metrics["face_width"]), 1e-6))
    if pose is not None:
        jaw_shift -= 0.12 * float(pose.get("yaw", 0.0))

    upper_up_left = max(clamp01((0.19 - float(metrics["left_upper_nose_gap"])) / 0.10), clamp01(-delta(metrics, neutral, "left_upper_nose_gap", 0.05)))
    upper_up_right = max(clamp01((0.19 - float(metrics["right_upper_nose_gap"])) / 0.10), clamp01(-delta(metrics, neutral, "right_upper_nose_gap", 0.05)))
    lower_down_left = max(jaw_open * 0.55 + mouth_frown_left * 0.25, clamp01(-delta(metrics, neutral, "left_lower_chin_gap", 0.07)))
    lower_down_right = max(jaw_open * 0.55 + mouth_frown_right * 0.25, clamp01(-delta(metrics, neutral, "right_lower_chin_gap", 0.07)))
    mouth_press_left = max(clamp01((0.045 - float(metrics["left_mouth_press_gap"])) / 0.03) * clamp01((0.030 - jaw_open) / 0.030), clamp01(-delta(metrics, neutral, "left_mouth_press_gap", 0.02)))
    mouth_press_right = max(clamp01((0.045 - float(metrics["right_mouth_press_gap"])) / 0.03) * clamp01((0.030 - jaw_open) / 0.030), clamp01(-delta(metrics, neutral, "right_mouth_press_gap", 0.02)))
    chin_drop_delta = clamp01(delta(metrics, neutral, "chin_drop", 0.12))
    mouth_close = clamp01((chin_drop_delta - jaw_open) * 1.6)
    mouth_roll_upper = max(clamp01((0.040 - float(metrics["upper_lip_thickness"])) / 0.025), clamp01(-delta(metrics, neutral, "upper_lip_thickness", 0.018))) * clamp01((0.030 - float(metrics["mouth_open_inner"])) / 0.030)
    mouth_roll_lower = max(clamp01((0.040 - float(metrics["lower_lip_thickness"])) / 0.025), clamp01(-delta(metrics, neutral, "lower_lip_thickness", 0.018))) * clamp01((0.030 - float(metrics["mouth_open_inner"])) / 0.030)
    mouth_shrug_upper = clamp01(upper_up_left * 0.5 + upper_up_right * 0.5 + mouth_close * 0.2)
    mouth_shrug_lower = clamp01(max(clamp01((float(metrics["left_lower_chin_gap"]) - 0.24) / 0.12), clamp01((float(metrics["right_lower_chin_gap"]) - 0.24) / 0.12)) * 0.7 + mouth_close * 0.2)
    cheek_puff = clamp01(mouth_pucker * 0.7 * (1.0 - jaw_open))
    cheek_squint_left = clamp01(0.5 * mouth_smile_left + 0.35 * (1.0 - left_wide_abs) + 0.2 * left_blink)
    cheek_squint_right = clamp01(0.5 * mouth_smile_right + 0.35 * (1.0 - right_wide_abs) + 0.2 * right_blink)
    nose_sneer_left = clamp01(0.55 * upper_up_left + 0.25 * mouth_smile_left + 0.15 * cheek_squint_left)
    nose_sneer_right = clamp01(0.55 * upper_up_right + 0.25 * mouth_smile_right + 0.15 * cheek_squint_right)

    shapes = {name: 0.0 for name in ARKIT_52_BLENDSHAPES}
    shapes.update({
        "browDownLeft": brow_down_left, "browDownRight": brow_down_right, "browInnerUp": brow_inner_up,
        "browOuterUpLeft": left_outer_up, "browOuterUpRight": right_outer_up, "cheekPuff": cheek_puff,
        "cheekSquintLeft": cheek_squint_left, "cheekSquintRight": cheek_squint_right,
        "eyeBlinkLeft": left_blink, "eyeBlinkRight": right_blink,
        "eyeSquintLeft": clamp01(left_blink * 0.65 + cheek_squint_left * 0.25),
        "eyeSquintRight": clamp01(right_blink * 0.65 + cheek_squint_right * 0.25),
        "eyeWideLeft": max(left_wide_abs, clamp01(delta(metrics, neutral, "left_eye_open", 0.08))),
        "eyeWideRight": max(right_wide_abs, clamp01(delta(metrics, neutral, "right_eye_open", 0.08))),
        "jawForward": clamp01(max(mouth_pucker, mouth_funnel) * 0.25),
        "jawLeft": clamp01(max(-jaw_shift, 0.0) / 0.06), "jawOpen": jaw_open, "jawRight": clamp01(max(jaw_shift, 0.0) / 0.06),
        "mouthClose": mouth_close, "mouthDimpleLeft": clamp01(mouth_smile_left * 0.55 + mouth_stretch_left * 0.25),
        "mouthDimpleRight": clamp01(mouth_smile_right * 0.55 + mouth_stretch_right * 0.25),
        "mouthFrownLeft": mouth_frown_left, "mouthFrownRight": mouth_frown_right, "mouthFunnel": mouth_funnel,
        "mouthLeft": clamp01(max(-mouth_center_shift, 0.0) / 0.05), "mouthLowerDownLeft": clamp01(lower_down_left),
        "mouthLowerDownRight": clamp01(lower_down_right), "mouthPressLeft": mouth_press_left, "mouthPressRight": mouth_press_right,
        "mouthPucker": mouth_pucker, "mouthRight": clamp01(max(mouth_center_shift, 0.0) / 0.05),
        "mouthRollLower": mouth_roll_lower, "mouthRollUpper": mouth_roll_upper,
        "mouthShrugLower": mouth_shrug_lower, "mouthShrugUpper": mouth_shrug_upper,
        "mouthSmileLeft": mouth_smile_left, "mouthSmileRight": mouth_smile_right,
        "mouthStretchLeft": max(mouth_stretch_left, mouth_width_abs * 0.55),
        "mouthStretchRight": max(mouth_stretch_right, mouth_width_abs * 0.55),
        "mouthUpperUpLeft": upper_up_left, "mouthUpperUpRight": upper_up_right,
        "noseSneerLeft": nose_sneer_left, "noseSneerRight": nose_sneer_right,
    })

    if use_head_pose_as_eye_gaze and pose is not None:
        yaw = float(pose.get("yaw", 0.0))
        pitch = float(pose.get("pitch", 0.0))
        left_amount = clamp01(max(-yaw, 0.0) / 0.35)
        right_amount = clamp01(max(yaw, 0.0) / 0.35)
        up_amount = clamp01(max(-pitch, 0.0) / 0.25)
        down_amount = clamp01(max(pitch, 0.0) / 0.25)
        shapes["eyeLookInLeft"] = left_amount
        shapes["eyeLookOutLeft"] = right_amount
        shapes["eyeLookInRight"] = right_amount
        shapes["eyeLookOutRight"] = left_amount
        shapes["eyeLookUpLeft"] = up_amount
        shapes["eyeLookUpRight"] = up_amount
        shapes["eyeLookDownLeft"] = down_amount
        shapes["eyeLookDownRight"] = down_amount

    return {k: clamp01(v) for k, v in shapes.items()}


def should_update_neutral(shapes, pose):
    yaw = abs(float((pose or {}).get("yaw", 0.0)))
    pitch = abs(float((pose or {}).get("pitch", 0.0)))
    roll = abs(float((pose or {}).get("roll", 0.0)))
    if yaw > 0.45 or pitch > 0.35 or roll > 0.45:
        return False
    active = max(shapes["jawOpen"], shapes["mouthSmileLeft"], shapes["mouthSmileRight"], shapes["mouthFrownLeft"], shapes["mouthFrownRight"], shapes["eyeBlinkLeft"], shapes["eyeBlinkRight"], shapes["browInnerUp"], shapes["browDownLeft"], shapes["browDownRight"])
    return active < 0.28


def convert_sequence(outputs, mean_face, shape_basis, blendshape_basis, mirror_input=False, neutral_momentum=0.90, use_head_pose_as_eye_gaze=False):
    arr = np.asarray(outputs, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[None, :]
    if arr.ndim != 2 or arr.shape[1] != 264:
        raise ValueError(f"Expected (T, 264) or (264,), got {arr.shape}")

    all_shapes = []
    neutral = None
    momentum = float(np.clip(neutral_momentum, 0.0, 0.999))

    for frame in arr:
        points, pose = reconstruct_68(frame, mean_face, shape_basis, blendshape_basis)
        metrics = extract_metrics(points, mirror_input=mirror_input)
        if neutral is None:
            neutral = dict(metrics)
        shapes = metrics_to_shapes(metrics, neutral, pose, use_head_pose_as_eye_gaze=use_head_pose_as_eye_gaze)
        if should_update_neutral(shapes, pose):
            neutral = {k: momentum * float(neutral[k]) + (1.0 - momentum) * float(v) for k, v in metrics.items()}
        all_shapes.append(shapes)
    return all_shapes


def stack_shape_dicts(shape_dicts):
    return np.asarray([[frame[name] for name in ARKIT_52_BLENDSHAPES] for frame in shape_dicts], dtype=np.float32)


print("helper ready")


Writing /content/qualcomm_3dmm_to_arkit52.py


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files

from qualcomm_3dmm_to_arkit52 import (
    ARKIT_52_BLENDSHAPES,
    convert_sequence,
    stack_shape_dicts,
)

WORK_DIR = Path("/content/qualcomm_3dmm_to_arkit52")
WORK_DIR.mkdir(exist_ok=True)


def load_coeff_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".npy":
        arr = np.load(path)
    elif suffix == ".json":
        data = json.loads(path.read_text())
        if isinstance(data, dict):
            if "coeff" in data:
                data = data["coeff"]
            elif "coeffs" in data:
                data = data["coeffs"]
            elif "outputs" in data:
                data = data["outputs"]
        arr = np.asarray(data, dtype=np.float32)
    elif suffix == ".txt":
        arr = np.loadtxt(path, dtype=np.float32)
    else:
        raise ValueError(f"Unsupported coeff file: {path.name}")
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    elif arr.ndim > 2:
        arr = arr.reshape(-1, arr.shape[-1])
    if arr.shape[-1] != 264:
        raise ValueError(f"Expected last dim 264, got {arr.shape}")
    return arr


print("upload these 4 files:")
print("1) Qualcomm output (.npy/.json/.txt)")
print("2) meanFace.npy")
print("3) shapeBasis.npy")
print("4) blendShape.npy")
uploaded = files.upload()

for name, payload in uploaded.items():
    (WORK_DIR / name).write_bytes(payload)

coeff_path = None
for p in WORK_DIR.iterdir():
    low = p.name.lower()
    if low == "meanface.npy" or low == "shapebasis.npy" or low == "blendshape.npy":
        continue
    if p.suffix.lower() in {".npy", ".json", ".txt"}:
        coeff_path = p
        break

if coeff_path is None:
    raise FileNotFoundError("Qualcomm coeff file을 찾지 못했습니다.")

MEAN_FACE_PATH = WORK_DIR / "meanFace.npy"
SHAPE_BASIS_PATH = WORK_DIR / "shapeBasis.npy"
BLENDSHAPE_BASIS_PATH = WORK_DIR / "blendShape.npy"

assert MEAN_FACE_PATH.exists(), "meanFace.npy missing"
assert SHAPE_BASIS_PATH.exists(), "shapeBasis.npy missing"
assert BLENDSHAPE_BASIS_PATH.exists(), "blendShape.npy missing"

outputs = load_coeff_file(coeff_path)
mean_face = np.load(MEAN_FACE_PATH)
shape_basis = np.load(SHAPE_BASIS_PATH)
blendshape_basis = np.load(BLENDSHAPE_BASIS_PATH)

print("coeff:", coeff_path.name, outputs.shape)
print("assets:", mean_face.shape, shape_basis.shape, blendshape_basis.shape)


helper ready
upload these 4 files:
1) Qualcomm output (.npy/.json/.txt)
2) meanFace.npy
3) shapeBasis.npy
4) blendShape.npy


FileNotFoundError: Qualcomm coeff file을 찾지 못했습니다.

In [ ]:
MIRROR_INPUT = False
NEUTRAL_MOMENTUM = 0.90
USE_HEAD_POSE_AS_EYE_GAZE = False

shape_dicts = convert_sequence(
    outputs,
    mean_face,
    shape_basis,
    blendshape_basis,
    mirror_input=MIRROR_INPUT,
    neutral_momentum=NEUTRAL_MOMENTUM,
    use_head_pose_as_eye_gaze=USE_HEAD_POSE_AS_EYE_GAZE,
)
shape_array = stack_shape_dicts(shape_dicts)

json_path = WORK_DIR / "arkit52_blendshapes.json"
npy_path = WORK_DIR / "arkit52_blendshapes.npy"
csv_path = WORK_DIR / "arkit52_blendshapes.csv"

json_path.write_text(json.dumps(shape_dicts, ensure_ascii=False, indent=2))
np.save(npy_path, shape_array)
pd.DataFrame(shape_array, columns=ARKIT_52_BLENDSHAPES).to_csv(csv_path, index=False)

print("saved:")
print(json_path)
print(npy_path)
print(csv_path)
print("frames:", len(shape_dicts))
print("first frame sample:")
for name in ["jawOpen", "mouthSmileLeft", "mouthSmileRight", "eyeBlinkLeft", "eyeBlinkRight"]:
    print(name, round(shape_dicts[0][name], 4))


In [ ]:
preview = pd.DataFrame(shape_array, columns=ARKIT_52_BLENDSHAPES)
display(preview.head())


In [ ]:
files.download(str(WORK_DIR / "arkit52_blendshapes.json"))
# files.download(str(WORK_DIR / "arkit52_blendshapes.npy"))
# files.download(str(WORK_DIR / "arkit52_blendshapes.csv"))
